### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="hepatitis_survival_prediction",
    dataset_year="1981",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5Q59J",
    download_description="""
Get the UCI data.

wget https://archive.ics.uci.edu/static/public/46/hepatitis.zip && unzip hepatitis.zip hepatitis.data && rm hepatitis.zip && mkdir -p local-data-warehouse/hepatitis_survival_prediction && mv hepatitis.data local-data-warehouse/hepatitis_survival_prediction/
""",
    # References, note, this is the first mention of the dataset I found. But I think it is not the original source or year.
    academic_reference_bibtex="""@inproceedings{efron1981statistical,
  title={Statistical theory and the computer},
  author={Efron, Bradley and Gong, Gail},
  booktitle={Computer science and statistics: Proceedings of the 13th Symposium on the Interface},
  pages={3--7},
  year={1981},
  organization={Springer}
}
""",
    academic_reference_bibtex_key="efron1981statistical",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI.

- We encode missing values as NaN.
- We convert numeric features to float and categorical features to category dtype.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

columns = [
    "class",
    "age",
    "sex",
    "steroid",
    "antivirals",
    "fatigue",
    "malaise",
    "anorexia",
    "liver_big",
    "liver_firm",
    "spleen_palpable",
    "spiders",
    "ascites",
    "varices",
    "bilirubin",
    "alk_phosphate",
    "sgot",
    "albumin",
    "protime",
    "histology"
]

df = pd.read_csv(dataset_mold.path / "hepatitis.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.replace("?", np.nan)
as_float_type = ["age", "bilirubin", "alk_phosphate", "sgot", "albumin", "protime"]
as_cat_type = ["class", "sex", "steroid", "antivirals", "fatigue", "malaise", "anorexia", "liver_big", "liver_firm", "spleen_palpable", "spiders", "ascites", "varices", "histology"]
df[as_float_type] = df[as_float_type].astype(float)
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (155, 20)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 155
Columns: 20
Use sampling: False (sample size: 155)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['sgot', 'alk_phosphate', 'age', 'protime', 'bilirubin', 'albumin', 'liver_big', 'histology', 'antivirals', 'sex']
Rows remaining as candidates after top-10 filter: 0 (of 155)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,class,age,sex,steroid,antivirals,fatigue,malaise,anorexia,liver_big,liver_firm,spleen_palpable,spiders,ascites,varices,bilirubin,alk_phosphate,sgot,albumin,protime,histology
0,2,36.0,1,2,2,2,2,2,2,2,2,2,2,2,0.7,62.0,224.0,4.2,100.0,1
1,2,51.0,1,2,2,1,2,2,2,1,1,1,2,1,1.0,NaN,20.0,3.0,63.0,2
2,1,62.0,1,1,2,1,1,2,NaN,NaN,2,2,2,2,1.0,NaN,60.0,NaN,NaN,1
3,2,51.0,1,1,1,1,1,2,2,2,2,2,2,2,1.0,78.0,58.0,4.6,52.0,1
4,1,61.0,1,1,2,1,1,2,NaN,NaN,2,1,2,2,NaN,NaN,NaN,NaN,NaN,2


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,liver_firm,category,11.0,7.10,2.0,"2, 1"
1,liver_big,category,10.0,6.45,2.0,"2, 1"
2,spleen_palpable,category,5.0,3.23,2.0,"2, 1"
3,spiders,category,5.0,3.23,2.0,"2, 1"
4,ascites,category,5.0,3.23,2.0,"2, 1"
5,varices,category,5.0,3.23,2.0,"2, 1"
6,steroid,category,1.0,0.65,2.0,"2, 1"
7,fatigue,category,1.0,0.65,2.0,"1, 2"
8,malaise,category,1.0,0.65,2.0,"2, 1"
9,anorexia,category,1.0,0.65,2.0,"2, 1"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,155.0,41.200000,12.565878,7.0,78.0
bilirubin,149.0,1.427517,1.212149,0.3,8.0
alk_phosphate,126.0,105.325397,51.508109,26.0,295.0
sgot,151.0,85.894040,89.650890,14.0,648.0
albumin,139.0,3.817266,0.651523,2.1,6.4
protime,88.0,61.852273,22.875244,0.0,100.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                    
anorexia        1        2    122  78.71
                2        1     32  20.65
                3     <NA>      1   0.65
antivirals      1        2    131  84.52
                2        1     24  15.48
ascites         1        2    130  83.87
                2        1     20  12.90
                3     <NA>      5   3.23
class           1        2    123  79.35
                2        1     32  20.65
fatigue         1        1    100  64.52
                2        2     54  34.84
                3     <NA>      1   0.65
histology       1        1     85  54.84
                2        2     70  45.16
liver_big       1        2    120  77.42
                2        1     25  16.13
                3     <NA>     10   6.45
liver_firm      1        2     84  54.19
                2        1     60  38.71
                3     <NA>     11   7.10
malaise         1        2     93  60.00
                2        1     61  39.35
                3     <NA>      1   0.65
sex             1        1    139  89.68
                2        2     16  10.32
spiders         1        2     99  63.87
                2        1     51  32.90
                3     <NA>      5   3.23
spleen_palpable 1        2    120  77.42
                2        1     30  19.35
                3     <NA>      5   3.23
steroid         1        2     78  50.32
                2        1     76  49.03
                3     <NA>      1   0.65
varices         1        2    132  85.16
                2        1     18  11.61
                3     <NA>      5   3.23

In [8]:
# Target Distribution
target_df

,count,pct
class,,
2,123,79.35
1,32,20.65


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to hepatitis_survival_prediction/019d9ce4-8eba-7478-9a8a-9584b8f802d8
019d9ce4-8eba-7478-9a8a-9584b8f802d8
2c4bfadb5f07cae07f4880641a06da9af453599b8ca5507e802d035f2dff47d5
